In [2]:
import os
import glob
import zipfile
from pathlib import Path

from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, LongType
from pyspark.sql.functions import col, struct, to_json, from_json, avg, count, countDistinct, desc, asc


In [4]:
# SECTION A - Create local SparkSession with Kafka package
spark = (
    SparkSession.builder
    .appName("Lab1_Local_Spark_Kafka")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)


26/03/28 23:33:56 WARN Utils: Your hostname, MacBook-Air-cua-PHAM-2.local resolves to a loopback address: 127.0.0.1; using 192.168.1.96 instead (on interface en0)
26/03/28 23:33:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/phamnguyenviettri/HK252/bigdata/bigdatalab/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/phamnguyenviettri/.ivy2/cache
The jars for the packages stored in: /Users/phamnguyenviettri/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e90be874-c6a4-413b-9190-3a902527b5fe;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in local-m2-cache
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven

Spark version: 3.5.1
Spark master: local[*]


In [5]:
# SECTION B - Try kagglehub download first, then fallback
DATA_DIR = None

try:
    import kagglehub

    # Dataset slug for MovieLens latest-small on Kaggle
    downloaded_path = kagglehub.dataset_download("grouplens/movielens-latest-small")
    p = Path(downloaded_path)
    print("kagglehub downloaded path:", p)

    if p.is_file() and p.suffix.lower() == ".zip":
        extract_dir = Path("./data_movielens")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(p, "r") as zf:
            zf.extractall(extract_dir)

        candidate = extract_dir / "ml-latest-small"
        if candidate.exists():
            DATA_DIR = str(candidate.resolve())
    else:
        # Try common structures
        if (p / "ml-latest-small").exists():
            DATA_DIR = str((p / "ml-latest-small").resolve())
        elif (p / "ratings.csv").exists():
            DATA_DIR = str(p.resolve())
        else:
            # Last fallback: search ratings.csv under downloaded path
            matches = list(p.rglob("ratings.csv"))
            if matches:
                DATA_DIR = str(matches[0].parent.resolve())

except Exception as e:
    print("kagglehub download failed:", str(e))

if DATA_DIR is None:
    DATA_DIR = str(Path("./ml-latest-small").resolve())
    print("Using fallback DATA_DIR:", DATA_DIR)
else:
    print("Using DATA_DIR:", DATA_DIR)


/Users/phamnguyenviettri/HK252/bigdata/bigdatalab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 971k/971k [00:01<00:00, 879kB/s]

Extracting files...
kagglehub downloaded path: /Users/phamnguyenviettri/.cache/kagglehub/datasets/grouplens/movielens-latest-small/versions/2
Using DATA_DIR: /Users/phamnguyenviettri/.cache/kagglehub/datasets/grouplens/movielens-latest-small/versions/2


In [6]:
# SECTION B - Read CSV files into Spark DataFrames
ratings_path = os.path.join(DATA_DIR, "ratings.csv")
movies_path = os.path.join(DATA_DIR, "movies.csv")
tags_path = os.path.join(DATA_DIR, "tags.csv")

for p in [ratings_path, movies_path, tags_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required file: {p}")

df_ratings = spark.read.option("header", True).option("inferSchema", True).csv(ratings_path)
df_movies = spark.read.option("header", True).option("inferSchema", True).csv(movies_path)
df_tags = spark.read.option("header", True).option("inferSchema", True).csv(tags_path)

print("Loaded DataFrames successfully.")
print("ratings rows:", df_ratings.count())
print("movies rows:", df_movies.count())
print("tags rows:", df_tags.count())


Loaded DataFrames successfully.
ratings rows: 100836
movies rows: 9742
tags rows: 3683


In [7]:
# Show sample rows
print("=== ratings sample ===")
df_ratings.show(5, truncate=False)

print("=== movies sample ===")
df_movies.show(5, truncate=False)

print("=== tags sample ===")
df_tags.show(5, truncate=False)


=== ratings sample ===
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
+------+-------+------+---------+
only showing top 5 rows

=== movies sample ===
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|

In [8]:
# Print schema
print("=== ratings schema ===")
df_ratings.printSchema()

print("=== movies schema ===")
df_movies.printSchema()

print("=== tags schema ===")
df_tags.printSchema()


=== ratings schema ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

=== movies schema ===
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

=== tags schema ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)



## SECTION C — Start local Kafka workflow

Prerequisite:
- Kafka must be running at `localhost:9092`.

We will:
1. Create topics `ratings`, `movies`, `tags`.
2. Ignore "already exists" errors.
3. Publish Spark DataFrames to Kafka as JSON (`value` field).


In [9]:
# SECTION C - Create Kafka topics gracefully
KAFKA_BOOTSTRAP = "localhost:9092"
TOPICS = ["ratings", "movies", "tags"]

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP, client_id="lab1-admin")

new_topics = [NewTopic(name=t, num_partitions=1, replication_factor=1) for t in TOPICS]

try:
    admin.create_topics(new_topics=new_topics, validate_only=False)
    print("Topics created:", TOPICS)
except TopicAlreadyExistsError:
    print("Some or all topics already exist. Continuing...")
except Exception as e:
    print("Topic creation warning:", str(e))
finally:
    admin.close()


Topics created: ['ratings', 'movies', 'tags']


In [10]:
# SECTION C - Publish DataFrames to Kafka (batch write)
def publish_df_to_kafka(df, topic, key_col="movieId"):
    prepared = df.selectExpr(
        f"CAST({key_col} AS STRING) AS key",
        "to_json(struct(*)) AS value"
    )

    # Batch write to Kafka topic
    prepared.write.format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
        .option("topic", topic) \
        .save()

    print(f"Published topic '{topic}' with {df.count()} records.")

publish_df_to_kafka(df_ratings, "ratings", key_col="movieId")
publish_df_to_kafka(df_movies, "movies", key_col="movieId")
publish_df_to_kafka(df_tags, "tags", key_col="movieId")


Published topic 'ratings' with 100836 records.
Published topic 'movies' with 9742 records.
Published topic 'tags' with 3683 records.


## SECTION D — Exercise Task 1

Now we read back data from Kafka (batch mode), inspect raw Kafka schema, then:
- cast `value` to STRING
- parse JSON with **explicit schemas** using `from_json`
- create:
  - `df_ratings_parsed`
  - `df_movies_parsed`
  - `df_tags_parsed`


In [11]:
# SECTION D - Read raw Kafka data for each topic (batch)
def read_topic_raw(topic):
    return (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("endingOffsets", "latest")
        .load()
    )

df_ratings_kafka_raw = read_topic_raw("ratings")
df_movies_kafka_raw = read_topic_raw("movies")
df_tags_kafka_raw = read_topic_raw("tags")

print("=== Raw Kafka schema (same shape for each topic) ===")
df_ratings_kafka_raw.printSchema()

print("=== Raw ratings from Kafka ===")
df_ratings_kafka_raw.show(5, truncate=False)


=== Raw Kafka schema (same shape for each topic) ===
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

=== Raw ratings from Kafka ===


26/03/28 23:41:34 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+---------+------+-----------------------+-------------+
|key    |value                                                                                                                                                                                |topic  |partition|offset|timestamp              |timestampType|
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+---------+------+-----------------------+-------------+
|[31]   |[7B 22 75 73 65 72 49 64 22 3A 31 2C 22 6D 6F 76 69 65 49 64 22 3A 31 2C 22 72 61 74 69 6E 67 22 3A 34 2E 30 2C 22 74 69 6D 65 73 74 61 6D 70 22 3A 39 36 34 39 38 32 37 30 33 7D]   |ratings|0        |0     |2026-03-28 23:41:07

In [12]:
# SECTION D - Explicit JSON schemas
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True),
])

# Cast value to STRING and parse JSON using explicit schemas
df_ratings_parsed = (
    df_ratings_kafka_raw
    .selectExpr("CAST(value AS STRING) AS value_str")
    .select(from_json(col("value_str"), ratings_schema).alias("data"))
    .select("data.*")
)

df_movies_parsed = (
    df_movies_kafka_raw
    .selectExpr("CAST(value AS STRING) AS value_str")
    .select(from_json(col("value_str"), movies_schema).alias("data"))
    .select("data.*")
)

df_tags_parsed = (
    df_tags_kafka_raw
    .selectExpr("CAST(value AS STRING) AS value_str")
    .select(from_json(col("value_str"), tags_schema).alias("data"))
    .select("data.*")
)

print("Parsed DataFrames ready:")
print("ratings:", df_ratings_parsed.count(), "rows")
print("movies:", df_movies_parsed.count(), "rows")
print("tags:", df_tags_parsed.count(), "rows")


Parsed DataFrames ready:


26/03/28 23:41:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:41:56 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


ratings: 100836 rows


26/03/28 23:41:57 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


movies: 9742 rows
tags: 3683 rows


In [13]:
# Verify parsed outputs
print("=== df_ratings_parsed ===")
df_ratings_parsed.printSchema()
df_ratings_parsed.show(5, truncate=False)

print("=== df_movies_parsed ===")
df_movies_parsed.printSchema()
df_movies_parsed.show(5, truncate=False)

print("=== df_tags_parsed ===")
df_tags_parsed.printSchema()
df_tags_parsed.show(5, truncate=False)


=== df_ratings_parsed ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)



26/03/28 23:42:03 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
+------+-------+------+---------+
only showing top 5 rows

=== df_movies_parsed ===
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



26/03/28 23:42:03 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows

=== df_tags_parsed ===
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: long (nullable =

26/03/28 23:42:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+------+-------+---------------+----------+
|userId|movieId|tag            |timestamp |
+------+-------+---------------+----------+
|2     |60756  |funny          |1445714994|
|2     |60756  |Highly quotable|1445714996|
|2     |60756  |will ferrell   |1445714992|
|2     |89774  |Boxing story   |1445715207|
|2     |89774  |MMA            |1445715200|
+------+-------+---------------+----------+
only showing top 5 rows



## SECTION E — Exercise Task 2

Compute top 5 movies by average rating:
- include only movies with `rating_count > 30`
- output columns:
  - `movieId`
  - `title`
  - `avg_rating`
  - `rating_count`
- sort correctly


In [14]:
# SECTION E - Top 5 movies by average rating (count > 30)
df_top5_movies = (
    df_ratings_parsed
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("*").alias("rating_count")
    )
    .filter(col("rating_count") > 30)
    .join(df_movies_parsed.select("movieId", "title"), on="movieId", how="left")
    .select("movieId", "title", "avg_rating", "rating_count")
    .orderBy(desc("avg_rating"), desc("rating_count"), asc("movieId"))
    .limit(5)
)

df_top5_movies.show(truncate=False)


26/03/28 23:42:25 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:42:25 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-------+--------------------------------+-----------------+------------+
|movieId|title                           |avg_rating       |rating_count|
+-------+--------------------------------+-----------------+------------+
|318    |Shawshank Redemption, The (1994)|4.429022082018927|317         |
|1204   |Lawrence of Arabia (1962)       |4.3              |45          |
|858    |Godfather, The (1972)           |4.2890625        |192         |
|2959   |Fight Club (1999)               |4.272935779816514|218         |
|1276   |Cool Hand Luke (1967)           |4.271929824561403|57          |
+-------+--------------------------------+-----------------+------------+



## SECTION F — Exercise Task 3

Compute 5 worst tags associated with lowest average rating.
Output:
- `tag`
- `avg_rating`


In [15]:
# SECTION F - 5 worst tags by average rating
df_tag_rating = (
    df_tags_parsed.alias("t")
    .join(df_ratings_parsed.alias("r"), on="movieId", how="inner")
)

df_worst_tags = (
    df_tag_rating
    .groupBy("tag")
    .agg(avg("rating").alias("avg_rating"))
    .orderBy(asc("avg_rating"), asc("tag"))
    .limit(5)
)

df_worst_tags.show(truncate=False)


26/03/28 23:42:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:42:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------+------------------+
|tag     |avg_rating        |
+--------+------------------+
|symbolic|0.5               |
|shark   |1.4166666666666667|
|stage   |1.75              |
|Tokyo   |2.0               |
|SNL     |2.1               |
+--------+------------------+



## SECTION G — Exercise Task 4

Analyze whether these worst tags truly imply low-rated movies by checking:
- average rating for each tag
- number of distinct movies carrying that tag
- total number of tag occurrences


In [16]:
# SECTION G - Analysis for worst tags
df_worst_tag_analysis = (
    df_tag_rating
    .join(df_worst_tags.select("tag"), on="tag", how="inner")
    .groupBy("tag")
    .agg(
        avg("rating").alias("avg_rating"),
        countDistinct("movieId").alias("distinct_movies"),
        count("*").alias("occurrences")
    )
    .orderBy(asc("avg_rating"), desc("occurrences"))
)

df_worst_tag_analysis.show(truncate=False)


26/03/28 23:43:13 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:43:13 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:43:14 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/28 23:43:14 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------+------------------+---------------+-----------+
|tag     |avg_rating        |distinct_movies|occurrences|
+--------+------------------+---------------+-----------+
|symbolic|0.5               |1              |1          |
|shark   |1.4166666666666667|1              |6          |
|stage   |1.75              |1              |2          |
|Tokyo   |2.0               |1              |1          |
|SNL     |2.1               |1              |10         |
+--------+------------------+---------------+-----------+

